<a href="https://colab.research.google.com/github/tanviengineer/100-Day-PYTHON/blob/main/datprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1 . Initialze Spark Session

from pyspark.sql import SparkSession
from pyspark.sql.functions import col,count, when , unix_timestamp
spark = SparkSession.builder.appName("BigDataPractical2").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark Session Created Successfully ")

Spark Session Created Successfully 


In [ ]:
# 2.Load Large Dataset
#  Replace file path with your dataset path
file_path ="/content/Students Performance.csv"

df=spark.read.format("csv").option("header","true").option("inferSchema","true").load(file_path)
print("Schema :")
df.printSchema()
print("Total Rows : ", df.count())

Schema :
root
 |-- Math_Score: integer (nullable = true)
 |-- Reading_Score: integer (nullable = true)
 |-- Writing_Score: integer (nullable = true)
 |-- Placement_Score: integer (nullable = true)
 |-- Club_Join_Date: integer (nullable = true)

Total Rows :  399


In [ ]:
# 2.Load Large Dataset
#  Replace file path with your dataset path
file_path ="/content/Students Performance.csv"

df=spark.read.format("csv").option("header","true").option("inferSchema","true").load(file_path)
print("Schema :")
df.printSchema()
print("Total Rows : ", df.count())

Schema :
root
 |-- Math_Score: integer (nullable = true)
 |-- Reading_Score: integer (nullable = true)
 |-- Writing_Score: integer (nullable = true)
 |-- Placement_Score: integer (nullable = true)
 |-- Club_Join_Date: integer (nullable = true)

Total Rows :  399


In [ ]:
# 3. check missing value
print("\nMissing Values per Colume :")
missing_df = df.select([count (when(col(c).isNull(),c)).alias(c)for c in df.columns])
missing_df.show()


Missing Values per Colume :


NameError: name 'df' is not defined

In [ ]:
# 4. handling missing values
#  drop rows where fare _amount or trip_distance is missing
df= df.dropna(subset=["Writing_Score","Reading_Score"])
#  fill missing math_score with median
median_value = df.approxQuantile("Math_Score",[0.5],0.01)[0]
df=df.fillna ({"Math_Score": median_value})
print("Missing values handled . ")

Missing values handled . 


In [ ]:
#  5. Remove Outliers
# remove negative or zero values
df= df.filter((col("Writing_Score")>0)&(col("Reading_Score")>0))
# df.show()

#  IOR Methed for face_amount
quantiles=df.approxQuantile("Writing_Score",[0.25,0.75],0.01)
Q1,Q3 = quantiles
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
df = df.filter((col("Writing_Score")>=lower_bound)&(col("Writing_Score")<=upper_bound))
print("Outliers removed using IQR")


Outliers removed using IQR


In [ ]:
# 6 . Noise removal (Reading_Score )
# Apply IQR method for Reading_Score to remove outliers
quantiles_rs = df.approxQuantile("Reading_Score",[0.25,0.75],0.01)
Q1_rs, Q3_rs = quantiles_rs

#  using formula ---outliers remove using IQR method-----
IQR_rs = Q3_rs - Q1_rs
lower_bound_rs = Q1_rs - 1.5 * IQR_rs
upper_bound_rs = Q3_rs + 1.5 * IQR_rs
df = df.filter((col("Reading_Score") >= lower_bound_rs) & (col("Reading_Score") <= upper_bound_rs))
print("Noise (outliers) removed from Reading_Score using IQR")

Noise (outliers) removed from Reading_Score using IQR


In [ ]:
#  7. parallel processing Example
print("\n Number of Partitions : ", df.rdd.getNumPartitions())
df=df.repartition(8)
print("Repartitioned to 8 partition.")


 Number of Partitions :  8
Repartitioned to 8 partition.


In [ ]:
 # 8.
print("\n Number of Partitions : ", df.rdd.getNumPartitions())
df=df.repartition(8)
print("Repartitioned to 8 partition .")


 Number of Partitions :  8
Repartitioned to 8 partition .
